# Geometry‑Limited Propulsion — HLV‑AILEE Simulation Notebook

Reference simulations for the coherence‑gated propulsion model (HLV‑AILEE) from  
*Geometry‑Limited Propulsion: An HLV‑AILEE Formulation of Coherence‑Gated Momentum Transfer*.

This notebook visualizes:
- coherence gate vs phase deviation
- hysteresis in the phase window
- Δv vs power under different misalignments
- temporal and spectral Δφ proxies
- time‑varying gate effects on cumulative Δv

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))  # make src/ importable

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from coherence_gate import (
    coherence_gate,
    classical_delta_v,
    hlv_ailee_delta_v,
    gate_vs_phase_sweep,
    hysteresis_sweep,
)
from phase_alignment_metrics import (
    delta_phi_temporal,
    delta_phi_spectral,
    composite_delta_phi,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Imports OK.')

---
## 1. Gate shape: G(Δφ) for different α values

The coherence gate is:

$$G(t) = e^{-\alpha\,\Delta\phi(t)^2}$$

Larger α → narrower acceptance window → sharper collapse.

In [ ]:
phi_range = np.linspace(0, 2.5, 400)
alphas = [0.5, 1.0, 2.0, 5.0]

fig, ax = plt.subplots(figsize=(7, 4))
for alpha in alphas:
    phi, G = gate_vs_phase_sweep(phi_range, alpha=alpha)
    ax.plot(phi, G, label=f'α = {alpha}')

ax.axhline(0.5, color='grey', lw=0.8, ls='--', label='50 % coupling')
ax.set_xlabel('Phase deviation Δφ')
ax.set_ylabel('Gate G(Δφ)')
ax.set_title('Coherence gate: acceptance-window shape')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2. Hysteresis loop (Figure 2 of paper)

Up-sweeps collapse coupling at a smaller Δφ than down-sweeps recover it.  
Modelled here by different effective α values on each branch.

In [ ]:
phi_up   = np.linspace(0, 2.5, 200)
phi_down = np.linspace(2.5, 0, 200)

hyst = hysteresis_sweep(
    phi_up,   phi_down,
    alpha_up=3.0,    # steeper collapse
    alpha_down=1.5,  # shallower recovery
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hyst['phi_up'],   hyst['gate_up'],   label='Up-sweep (collapse)',   color='steelblue')
ax.plot(hyst['phi_down'], hyst['gate_down'], label='Down-sweep (recovery)', color='tomato',  ls='--')

# shade hysteresis region
phi_common = np.linspace(0, 2.5, 200)
G_up_interp   = np.interp(phi_common, hyst['phi_up'],   hyst['gate_up'])
G_down_interp = np.interp(phi_common, hyst['phi_down'][::-1], hyst['gate_down'][::-1])
ax.fill_between(phi_common, G_up_interp, G_down_interp,
                alpha=0.15, color='purple', label='Hysteresis window')

ax.set_xlabel('Phase deviation Δφ')
ax.set_ylabel('Effective coupling G_eff')
ax.set_title('Phase window with hysteresis (Figure 2 of paper)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Δv vs. input power: gate-limited saturation

HLV–AILEE prediction: beyond a misalignment threshold, extra power **cannot**
compensate — a distinguishing signature vs. classical smooth saturation.

We compare three cases:
- Classical (no gate, Δφ = 0)
- HLV–AILEE with low misalignment (Δφ = 0.3)
- HLV–AILEE with high misalignment (Δφ = 1.2)

In [ ]:
# Shared trajectory parameters
N = 500
t = np.linspace(0, 100, N)          # [s]
mass = np.linspace(1000, 600, N)    # [kg]  propellant burn
velocity = np.linspace(100, 400, N) # [m/s] reference trajectory
ISP = 350.0                         # [s]
ETA = 0.85
ALPHA = 2.0

power_levels = np.linspace(1e4, 2e6, 30)  # [W]

dv_classical     = []
dv_low_mismatch  = []
dv_high_mismatch = []

for P in power_levels:
    p_in = np.full(N, P)

    # Classical: gate = 1 everywhere
    dv_classical.append(
        hlv_ailee_delta_v(t, p_in, mass, velocity,
                          delta_phi=np.zeros(N),
                          isp=ISP, eta=ETA, alpha=ALPHA)
    )
    # Low misalignment
    dv_low_mismatch.append(
        hlv_ailee_delta_v(t, p_in, mass, velocity,
                          delta_phi=np.full(N, 0.3),
                          isp=ISP, eta=ETA, alpha=ALPHA)
    )
    # High misalignment
    dv_high_mismatch.append(
        hlv_ailee_delta_v(t, p_in, mass, velocity,
                          delta_phi=np.full(N, 1.2),
                          isp=ISP, eta=ETA, alpha=ALPHA)
    )

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(power_levels / 1e3, dv_classical,     label='Δφ = 0 (classical)', color='grey')
ax.plot(power_levels / 1e3, dv_low_mismatch,  label='Δφ = 0.3 (low)',     color='steelblue')
ax.plot(power_levels / 1e3, dv_high_mismatch, label='Δφ = 1.2 (high)',    color='tomato')

ax.set_xlabel('Input power P_in [kW]')
ax.set_ylabel('Δv  [m s⁻¹]')
ax.set_title('Gate-limited saturation: Δv vs. power for different misalignments')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Gate suppression at Δφ=0.3: {coherence_gate(0.3, ALPHA):.3f}')
print(f'Gate suppression at Δφ=1.2: {coherence_gate(1.2, ALPHA):.3f}')

---
## 4. Phase-proxy demonstration

### 4a. Temporal proxy Δφ_temp  (Eq. 7)

$$\Delta\phi_{\text{temp}} \sim \frac{|f_{\text{sys}} - f_{\text{lat}}|}{f_{\text{lat}}}$$

We simulate a system frequency that drifts upward over the burn duration.

In [ ]:
rng = np.random.default_rng(42)

f_lat = 100.0  # reference lattice frequency [Hz]
# Simulate a drifting system frequency over time
f_sys_time = f_lat + np.linspace(0, 30, N) + rng.normal(0, 1, N)

phi_temp = delta_phi_temporal(f_sys_time, f_lat)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(t, f_sys_time, color='steelblue', lw=0.8)
axes[0].axhline(f_lat, color='grey', ls='--', label=f'f_lat = {f_lat} Hz')
axes[0].set_ylabel('Frequency [Hz]')
axes[0].set_title('System frequency drift')
axes[0].legend(fontsize=8)

axes[1].plot(t, phi_temp, color='tomato', lw=0.8)
axes[1].set_ylabel('Δφ_temp')
axes[1].set_xlabel('Time [s]')
axes[1].set_title('Temporal proxy Δφ_temp (Eq. 7)')
plt.tight_layout()
plt.show()

### 4b. Spectral proxy Δφ_spec  (Eq. 9)

$$\Delta\phi_{\text{spec}} \sim \int d\omega\,|S_{\text{thrust}}(\omega) - S_{\text{lat}}(\omega)|$$

We compare a 50 Hz thrust tone against a 50 Hz reference (aligned) and
a 90 Hz reference (misaligned).  Requires SciPy; skipped gracefully if absent.

In [ ]:
try:
    from scipy.signal import welch
    SCIPY_OK = True
except ImportError:
    SCIPY_OK = False
    print('SciPy not found — skipping spectral proxy demo.')

if SCIPY_OK:
    SR = 1024.0  # sample rate [Hz]
    t_sig = np.linspace(0, 2, int(2 * SR), endpoint=False)
    noise = rng.normal(0, 0.05, len(t_sig))

    thrust_signal  = np.sin(2 * np.pi * 50 * t_sig) + noise
    lat_aligned    = np.sin(2 * np.pi * 50 * t_sig)  # same frequency
    lat_misaligned = np.sin(2 * np.pi * 90 * t_sig)  # different frequency

    phi_spec_aligned    = delta_phi_spectral(thrust_signal, lat_aligned,    SR)
    phi_spec_misaligned = delta_phi_spectral(thrust_signal, lat_misaligned, SR)

    print(f'Δφ_spec (50 Hz vs 50 Hz, aligned):    {phi_spec_aligned:.4f}')
    print(f'Δφ_spec (50 Hz vs 90 Hz, misaligned): {phi_spec_misaligned:.4f}')

    # Plot PSDs
    freqs_t, S_t = welch(thrust_signal,  fs=SR)
    freqs_a, S_a = welch(lat_aligned,    fs=SR)
    freqs_m, S_m = welch(lat_misaligned, fs=SR)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(freqs_t, S_t, label='Thrust (50 Hz)',             color='steelblue')
    ax.semilogy(freqs_a, S_a, label='Lattice aligned (50 Hz)',    color='green',  ls='--')
    ax.semilogy(freqs_m, S_m, label='Lattice misaligned (90 Hz)', color='tomato', ls=':')
    ax.set_xlim(0, 200)
    ax.set_xlabel('Frequency [Hz]')
    ax.set_ylabel('PSD')
    ax.set_title('Spectral proxy: PSD comparison')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

---
## 5. Full integration: gated Δv over a time-varying trajectory

We combine a time-varying Δφ (rising quadratically over the burn) with the
full HLV–AILEE integral to show gate suppression accumulating in real time.
A composite Δφ is also computed by blending the trajectory proxy with a
placeholder spatial proxy.

In [ ]:
# Slowly drifting phase deviation over the burn
phi_trajectory = 0.05 + 0.8 * (t / t[-1])**2  # rises from ~0.05 to ~0.85

gate_trajectory = coherence_gate(phi_trajectory, alpha=ALPHA)

p_in_const = np.full(N, 5e5)  # 500 kW constant

dv_gated   = hlv_ailee_delta_v(t, p_in_const, mass, velocity,
                                phi_trajectory, ISP, ETA, ALPHA)
dv_ungated = hlv_ailee_delta_v(t, p_in_const, mass, velocity,
                                np.zeros(N), ISP, ETA, ALPHA)

# Composite Δφ using temporal + constant spatial proxy
phi_composite = composite_delta_phi(
    phi_trajectory,
    np.full(N, 0.1),   # placeholder spatial proxy
    weights=[0.7, 0.3]
)

fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)

axes[0].plot(t, phi_trajectory, color='darkorange', label='Δφ(t)')
axes[0].plot(t, phi_composite,  color='purple',     label='Δφ_composite', ls='--')
axes[0].set_ylabel('Phase deviation')
axes[0].set_title('Time-varying phase deviation')
axes[0].legend(fontsize=8)

axes[1].plot(t, gate_trajectory, color='steelblue')
axes[1].axhline(1.0, color='grey', lw=0.8, ls='--', label='No gate (classical)')
axes[1].set_ylabel('Gate G(t)')
axes[1].set_title('Coherence gate over burn')
axes[1].legend(fontsize=8)

# Running integral of gated vs ungated coupling
integrand_gated   = (p_in_const / (mass * velocity)) * gate_trajectory
integrand_ungated = (p_in_const / (mass * velocity))
cumulative_gated   = ISP * ETA * np.array([np.trapz(integrand_gated[:i+1],   t[:i+1]) for i in range(N)])
cumulative_ungated = ISP * ETA * np.array([np.trapz(integrand_ungated[:i+1], t[:i+1]) for i in range(N)])

axes[2].plot(t, cumulative_gated,   color='tomato', label=f'HLV–AILEE  Δv = {dv_gated:.1f} m/s')
axes[2].plot(t, cumulative_ungated, color='grey',   label=f'Classical   Δv = {dv_ungated:.1f} m/s', ls='--')
axes[2].set_ylabel('Cumulative Δv [m s⁻¹]')
axes[2].set_xlabel('Time [s]')
axes[2].set_title('Cumulative velocity gain: gated vs. ungated')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'\nFinal Δv (HLV–AILEE, gated):   {dv_gated:.2f} m/s')
print(f'Final Δv (classical, Δφ=0):    {dv_ungated:.2f} m/s')
print(f'Gate suppression factor:        {dv_gated / dv_ungated:.3f}')

---
## 6. Next steps for experimenters

| Protocol phase | Actions |
|---|---|
| **Baseline** | Measure thrust, Isp, power, spectra; log geometry & environment |
| **Perturbations** | Apply timing offsets; modulate geometry; log ignition/shutdown history |
| **Analysis** | Fit α from threshold data; plot hysteresis loops; compare null baselines |
| **Replication** | Pre-register metrics; share raw data and settings |

**Falsification criteria** (Section 6.1 of paper):

1. No measurable correlation between Δφ proxies and coupling efficiency → **falsified**
2. Increased power fully compensates misalignment → **falsified**
3. Phase drift produces no threshold-like degradation or hysteresis → **falsified**

---
*End of simulation notebook.*